In [ ]:
# 进口

import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

# 如果运行此单元时出现错误，请转到故障排除笔记本！

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 在这里 - 请参阅 base_url

openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')



In [ ]:
# 代表网页的类
# 如果您不熟悉类，请查看“中级 Python”笔记本

# 有些网站需要您在获取时使用正确的标头：
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

class Website:

    def __init__(self, url):
        """
        Create this Website object from the given url using the BeautifulSoup library
        """
        self.url = url
        response = requests.get(url, headers=headers)
        soup = BeautifulSoup(response.content, 'html.parser')
        self.title = soup.title.string if soup.title else "No title found"
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        self.text = soup.body.get_text(separator="\n", strip=True)

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 我们来尝试一下。更改网站并添加打印声明以进行后续操作。

ed = Website("https://marc-views.vercel.app/")
print(ed.title)
print(ed.text)

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 定义我们的系统提示符 - 您可以稍后进行实验，将最后一句更改为“用西班牙语以 markdown 方式回复”。

system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 编写询问网站摘要的用户提示的函数：

def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [ ]:
print(user_prompt_for(ed))

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "What is 2 + 2?"}
]

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 为您提供预览 - 使用系统和用户消息调用 OpenAI：

response = openai.chat.completions.create(model="llama3.2", messages=messages)
print(response.choices[0].message.content)

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 看看这个函数如何创建与上面完全相同的格式

def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 尝试一下，然后再尝试几个网站

messages_for(ed)

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 现在：调用 OpenAI API。你将会对此非常熟悉！

def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "llama3.2",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
summarize("https://marc-views.vercel.app/")

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
# 使用 markdown 在 Jupyter 输出中很好地显示此内容的函数

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
# 小白提示：下面代码逻辑未改，仅补充中文注释便于阅读
display_summary("https://marc-views.vercel.app/")

In [ ]:
# 第 1 步：创建提示

system_prompt = "You are a helpful assitant, mindful about the user which seems not so good at speaking english, because its his second language. You are trying to suggest an appropriate and impactful subject line for the email content provided by the User!"
user_prompt = """
    Hii I am rava, I am writing this email so I ask you if you remember that you told me to come to interview tomorrow only if i have AI experience already, the position is AI engineer, but i have been only knowing pyhton, working into full stack position. Forgive my english if you think i am not able to speak, then i should tell you, my english might broke, but i will not. I am studying, working, never lacking the courage and attitude to show that knowing less is not a flaw, wanting not to learn is, so I am telling you through this email, you will be happy to offering me this chance.
"""

# 第 2 步：创建消息列表

messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}] # fill this in

# 第三步：调用OpenAI

response = openai.chat.completions.create(
    model= "llama3.2",
    messages=messages
)

# 第四步：打印结果

print(response.choices[0].message.content)